# Q30: Bi-encoders vs Cross-encoders
**Topic:** LLM | **Difficulty:** Medium | **Time:** 20 min
**Sheet:** GrindLLM50

---

## Question
How are Bi-encoders and cross-encoders trained? Which is used where?

## Detailed Answer

### Bi-encoder
Encodes query and document **independently** into fixed-size vectors.
```
Query → Encoder A → q_emb ⟍
                            → cosine_sim(q_emb, d_emb) → score
Doc   → Encoder B → d_emb ⟋
```

**Training**:
- Contrastive loss (InfoNCE): Pull positive pairs together, push negatives apart
- Triplet loss: $L = \max(0, d(q, p) - d(q, n) + m)$
- In-batch negatives: Use other batch examples as negatives
- Hard negative mining: Use BM25 or previous model for hard negatives

### Cross-encoder
Processes query and document **jointly**.
```
[CLS] query [SEP] document [SEP] → Encoder → [CLS] → Linear → score
```

**Training**:
- Binary cross-entropy: Is this pair relevant (1) or not (0)?
- Pairwise ranking: Score(q, pos) > Score(q, neg)
- Dataset: (query, document, label) triples

### Key Differences
| Aspect | Bi-encoder | Cross-encoder |
|--------|-----------|---------------|
| Encoding | Independent | Joint |
| Speed | Fast (pre-compute docs) | Slow (per-pair) |
| Quality | Good | Better |
| Pre-compute docs | Yes (offline) | No |
| Use case | Retrieval | Re-ranking |
| Scale | Millions of docs | Top 20-100 |

### Production Pattern
```
Query → Bi-encoder retrieval (ANN search, top-100)
      → Cross-encoder re-ranking (top-100 → top-10)
      → Return results
```

### Examples
- **Bi-encoder**: Sentence-BERT, E5, BGE, GTE, OpenAI embeddings
- **Cross-encoder**: ms-marco-MiniLM, Cohere Rerank, BGE-reranker

## Key Takeaways
- **Bi-encoder**: Independent encoding, fast retrieval over millions of docs
- **Cross-encoder**: Joint encoding, higher quality but O(N) forward passes
- Production: **bi-encoder (recall) → cross-encoder (precision)**
- Train bi-encoders with contrastive loss + hard negatives